In [1]:
# 04_separation_theorem.ipynb
# Theorem 4 (separation): the migration-to-reversal ratio grows like K, so
# grades destabilize while ranking stays robust. Left: M/R versus K at fixed
# Delta. Right: the separation window where AUC loss stays ~0 while a sizeable
# share of applicants migrate grade.

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from sklearn.metrics import roc_auc_score
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"savefig.dpi": 600, "figure.dpi": 110, "axes.grid": True,
                     "grid.color": "0.9", "font.size": 9, "axes.edgecolor": "0.3"})
GREYS = ["0.0", "0.3", "0.5", "0.65"]; LS = ["-", "--", "-.", ":"]; MK = ["o", "s", "^", "D"]
Path("../results/figures").mkdir(parents=True, exist_ok=True)
Path("../results/tables").mkdir(parents=True, exist_ok=True)
def _save(fig, name):
    fig.savefig(f"../results/figures/{name}.png", bbox_inches="tight")
    fig.savefig(f"../results/figures/{name}.pdf", bbox_inches="tight")

df = pd.read_parquet("../data/synthetic_population.parquet")
ccols = [c for c in df.columns if c.startswith("c")]
C = df[ccols].to_numpy(); f = df["f"].to_numpy(); p = df["p"].to_numpy(); y = df["y"].to_numpy()
P = len(ccols); b0 = float((f - C.sum(1)).mean())
def quant(A, D): return D * np.round(A / D)
def fq(D): return b0 + quant(C, D).sum(1)

Deltas = np.array([0.02, 0.05, 0.1, 0.2, 0.4])
Ks = [5, 7, 10, 20]
auc0 = roc_auc_score(y, f)

def migration(fc, fqv, K):
    q = np.quantile(fc, np.linspace(0, 1, K + 1)[1:-1])
    return float(np.mean(np.digitize(fc, q) != np.digitize(fqv, q)))

def reversal_rate(fc, fqv, npair=200000, seed=0):
    r = np.random.default_rng(seed)
    pos = np.where(y == 1)[0]; neg = np.where(y == 0)[0]
    i = r.choice(pos, npair); j = r.choice(neg, npair)
    return float(np.mean((np.sign(fc[i] - fc[j]) > 0) & (np.sign(fqv[i] - fqv[j]) <= 0)))

Dfix = 0.1; fqv = fq(Dfix); R = reversal_rate(f, fqv)
rows = [dict(K=K, M=migration(f, fqv, K), R=R, ratio=migration(f, fqv, K) / R) for K in Ks]
rtab = pd.DataFrame(rows)
rtab.to_csv("../results/tables/04_separation_ratio.csv", index=False)
print(rtab.round(3).to_string(index=False))

rows2 = []
for D in Deltas:
    fqv = fq(D)
    rows2.append(dict(Delta=D, auc_loss=auc0 - roc_auc_score(y, fqv),
                      migration_K10=migration(f, fqv, 10)))
wtab = pd.DataFrame(rows2)
wtab.to_csv("../results/tables/04_separation_window.csv", index=False)
print(wtab.round(4).to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
ax[0].plot(rtab.K, rtab.ratio, ls="-", color="0.0", marker="o")
ax[0].set_xlabel("number of bands K")
ax[0].set_ylabel(r"migration / reversal (at $\Delta$=0.1)")
l1 = ax[1].plot(wtab.Delta, wtab.migration_K10, ls="-", color="0.0", marker="o",
                label="band migration (K=10)")
ax[1].set_xlabel(r"lattice spacing $\Delta$"); ax[1].set_ylabel("band migration rate")
ax2 = ax[1].twinx()
l2 = ax2.plot(wtab.Delta, wtab.auc_loss, ls="--", color="0.55", marker="s", label="AUC loss")
ax2.set_ylabel("AUC loss"); ax2.grid(False)
ax[1].legend(handles=l1 + l2, fontsize=7, loc="upper left")
fig.tight_layout(); _save(fig, "04_separation_theorem"); plt.close(fig)

 K     M     R  ratio
 5 0.068 0.009  7.841
 7 0.095 0.009 10.961
10 0.140 0.009 16.143
20 0.287 0.009 33.051
 Delta  auc_loss  migration_K10
  0.02    0.0002         0.0272
  0.05    0.0001         0.0758
  0.10    0.0011         0.1402
  0.20    0.0029         0.2972
  0.40    0.0226         0.6783
